In [ ]:
# from modelscope.msdatasets import MsDataset
#
# dataset=MsDataset.load('../data/train.jsonl')
# data=dataset.to_hf_dataset().select(range(100))
# data[:10]


import json

data = []
target = 200
count = 0
with open('../data/train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        data.append(item)
        count += 1
        if count >= target:
            break

data[:10]

In [ ]:
from modelscope import AutoModelForCausalLM

model_name = "openai-community/gpt2"
gpt_model = AutoModelForCausalLM.from_pretrained(model_name)
print(gpt_model)

In [ ]:
from modelscope import AutoTokenizer

gpt_tokenizer = AutoTokenizer.from_pretrained(model_name)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
gpt_tokenizer.pad_token_id


In [ ]:
from torch.utils.data import Dataset, DataLoader

prompt = "问题:{}\n答案:"


class dataset(Dataset):
    def __init__(self, data, max_length=128):
        self.data = data
        self.encodings = []
        for qa in data:
            text = prompt.format(qa['question']) + qa['answer'] + gpt_tokenizer.eos_token
            encoded = gpt_tokenizer(text, max_length=max_length, truncation=True, padding='max_length',
                                    return_tensors='pt')
            inpus_ids = encoded['input_ids'].squeeze()
            self.encodings.append(inpus_ids)

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]


dataset = dataset(data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch in dataloader:
    print(batch)
    break


In [59]:
import torch
from torch import optim, nn

optimizer = optim.Adam(gpt_model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpt_model.to(device)

for _ in range(20):
    for input_ids in dataloader:
        optimizer.zero_grad()
        input_ids = input_ids.to(device)
        output = gpt_model(input_ids, labels=input_ids)
        loss = output.loss
        loss.backward()
        optimizer.step()
        print(loss.item())

0.0442361980676651
0.0526791550219059
0.055060844868421555
0.05779904127120972
0.06096205860376358
0.05702554062008858
0.059780221432447433
0.06913870573043823
0.058171480894088745
0.056098856031894684
0.05360140651464462
0.05749192833900452
0.07102271914482117
0.09425684809684753
0.051496800035238266
0.04604290798306465
0.06784775853157043
0.07636851817369461
0.058200862258672714
0.06894075870513916
0.11069083958864212
0.07531385868787766
0.06258919835090637
0.057978611439466476
0.0760078951716423
0.06411892920732498
0.09308386594057083
0.13992223143577576


KeyboardInterrupt: 

In [57]:
def generate(sentence, max_length=128):
    input_ids=gpt_tokenizer.encode(sentence,return_tensors='pt')
    output=gpt_model.generate(
        input_ids,
        max_length=max_length,
        pad_token_id=gpt_tokenizer.eos_token_id
    )
    return gpt_tokenizer.decode(output[0],skip_special_tokens=True)

input=prompt.format("今天天气如何?")
result=generate(input)
print(result[len(input):])


今天天气《天使》们天天天气 中国的天使念景景物油片和西古昆酱和西贵烹等分的道肉料，玄贵烹等。
